# Graph Constant Bottleneck — abhinav153/Trash_classifier

**Smell (`predict.py` line 9):** `tf.convert_to_tensor(img, dtype=tf.float32)` is called inside the inference loop, embedding each image as a graph constant when called repeatedly in TF1-graph mode. In training, the analogous smell is embedding the entire dataset as `tf.constant`, pinning it in memory for the session lifetime.

**Fix:** Use a `tf.data.Dataset` pipeline for both training and inference — data flows through the graph without being permanently pinned.

*Note: CIFAR-10 (5 selected classes) is used as a proxy for the 5-class TrashNet dataset.*

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS  = 10
EPOCHS  = 3
BATCH   = 64
N_CLASS = 5

# Use 5 CIFAR-10 classes as proxy for TrashNet's 5 categories
(x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
mask_tr = (y_tr.flatten() < N_CLASS)
mask_te = (y_te.flatten() < N_CLASS)
x_tr, y_tr = x_tr[mask_tr].astype('float32')/255.0, y_tr[mask_tr].flatten()
x_te, y_te = x_te[mask_te].astype('float32')/255.0, y_te[mask_te].flatten()
# Resize to 96x96 for MobileNetV2
x_tr = tf.image.resize(x_tr, [96, 96]).numpy()
x_te = tf.image.resize(x_te, [96, 96]).numpy()
y_tr_oh = to_categorical(y_tr, N_CLASS)
y_te_oh = to_categorical(y_te, N_CLASS)
print(f'Data: train={x_tr.shape}  test={x_te.shape}')

In [ ]:
def build_model():
    """MobileNetV2 transfer-learning head (matches abhinav153/Trash_classifier architecture)"""
    base = MobileNetV2(input_shape=(96,96,3), include_top=False, weights='imagenet')
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dropout(0.2)(x)
    out = Dense(N_CLASS, activation='softmax')(x)
    model = Model(base.input, out)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model
print('MobileNetV2 builder ready')

## BEFORE — Smell Active (full dataset embedded as tf.constant in graph)

In [ ]:
results_before = []

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()

    tracker = EmissionsTracker(
        project_name=f'TrashCls_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    # ❌ SMELL (mirrors predict.py line 9 pattern at training scale):
    # Entire training set pinned as tf.constant — permanently in graph memory.
    X_const = tf.constant(x_tr, dtype=tf.float32)       # ← smell
    Y_const = tf.constant(y_tr_oh, dtype=tf.float32)    # ← smell

    model = build_model()
    model.fit(X_const, Y_const,
              epochs=EPOCHS, batch_size=BATCH, verbose=1,
              validation_data=(x_te, y_te_oh))
    _, acc = model.evaluate(x_te, y_te_oh, verbose=0)

    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'accuracy': round(float(acc),4),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  acc={acc:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del X_const, Y_const, model; gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/abhinav153_Trash_classifier_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/abhinav153_Trash_classifier_before.csv')

## AFTER — Smell Fixed (tf.data.Dataset pipeline — no full-dataset constant)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()

    tracker = EmissionsTracker(
        project_name=f'TrashCls_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    # ✅ FIX: stream data through tf.data — nothing permanently pinned in graph
    ds_train = (tf.data.Dataset
                .from_tensor_slices((x_tr, y_tr_oh))
                .shuffle(5000).batch(BATCH).prefetch(tf.data.AUTOTUNE))
    ds_val   = (tf.data.Dataset
                .from_tensor_slices((x_te, y_te_oh))
                .batch(BATCH).prefetch(tf.data.AUTOTUNE))

    model = build_model()
    model.fit(ds_train, epochs=EPOCHS, verbose=1, validation_data=ds_val)
    _, acc = model.evaluate(ds_val, verbose=0)

    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'accuracy': round(float(acc),4),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  acc={acc:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del ds_train, ds_val, model; gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/abhinav153_Trash_classifier_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/abhinav153_Trash_classifier_after.csv')

In [ ]:
print('\n=== SUMMARY: abhinav153/Trash_classifier ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")